### What is happening in the next code?

The code finds the project root, creates the path to the `data/raw` folder, and checks whether it exists.  
Then it prints the project location and lists the dataset folders inside `data/raw`.

In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data found:", RAW.is_dir())
print("Dataset folders:", [p.name for p in RAW.iterdir() if p.is_dir()])

Project root: d:\Maktab_Sharif_Projects\traffic-vehicle-classification
Raw data found: True
Dataset folders: ['dataset1', 'datasetv2_TrainUclean']


### What is happening in the next code?

The code defines all dataset/source folders that should exist and builds each folder's path inside `data/raw`.  
It then checks every folder and prints **FOUND** if it exists or **MISSING** if it does not.

In [7]:
SOURCES = [
    ("dataset1", "train"),
    ("dataset1", "test"),
    ("dataset1", "unclean"),
    ("datasetv2_TrainUclean", "train"),
    ("datasetv2_TrainUclean", "unclean"),
]

for dataset, source in SOURCES:
    folder = RAW / dataset / source
    print(f"{dataset}/{source}: {'FOUND' if folder.is_dir() else 'MISSING'}")

dataset1/train: FOUND
dataset1/test: FOUND
dataset1/unclean: FOUND
datasetv2_TrainUclean/train: FOUND
datasetv2_TrainUclean/unclean: FOUND


### What is happening in the next code?

The code goes through each dataset/source folder and finds all subfolders, which represent the image classes.  
It then sorts and prints the class names for each dataset/source.

In [8]:
for dataset, source in SOURCES:
    folder = RAW / dataset / source
    classes = sorted(p.name for p in folder.iterdir() if p.is_dir())
    print(f"{dataset}/{source}: {classes}")

dataset1/train: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
dataset1/test: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
dataset1/unclean: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
datasetv2_TrainUclean/train: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
datasetv2_TrainUclean/unclean: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus']


### What is happening in the next code?

The code defines the valid class names and converts alternative names like `nissan` and `neysan` into `vanet`.  
It then checks every dataset for class names that are **not expected** and prints those unexpected labels.

In [9]:
EXPECTED_CLASSES = {
    "ambulance", "autobus", "kamyun", "kamyunet",
    "minibus", "savari", "taxi", "vanet",
}

ALIASES = {
    "neysan": "vanet",
    "nysan": "vanet",
    "nissan": "vanet",
}

for dataset, source in SOURCES:
    folder = RAW / dataset / source
    labels = {
        ALIASES.get(p.name.strip().lower(), p.name.strip().lower())
        for p in folder.iterdir() if p.is_dir()
    }
    unexpected = labels - EXPECTED_CLASSES
    print(f"{dataset}/{source}: unexpected labels = {sorted(unexpected)}")

dataset1/train: unexpected labels = []
dataset1/test: unexpected labels = []
dataset1/unclean: unexpected labels = []
datasetv2_TrainUclean/train: unexpected labels = []
datasetv2_TrainUclean/unclean: unexpected labels = []


### What is happening in the next code?

The code scans all images, checks whether they are valid/readable, fixes class-name aliases, and records each image's information.  
Finally, it creates a DataFrame and reports the number of readable/unreadable images and the available classes.

In [10]:
from PIL import Image
import pandas as pd

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
rows = []
unreadable = []

for dataset, source in SOURCES:
    source_dir = RAW / dataset / source

    for path in sorted(source_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
            continue

        relative = path.relative_to(source_dir)
        if len(relative.parts) < 2:
            raise ValueError(f"Image is outside a class folder: {path}")

        original_label = relative.parts[0].strip().lower()
        label = ALIASES.get(original_label, original_label)

        try:
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                width, height = image.size
        except Exception as exc:
            unreadable.append((str(path), str(exc)))
            continue

        rows.append({
            "path": str(path),
            "dataset": dataset,
            "source": source,
            "original_label": original_label,
            "label": label,
            "width": width,
            "height": height,
        })

df = pd.DataFrame(rows)

print("Readable images:", len(df))
print("Unreadable images:", len(unreadable))
print("Classes:", sorted(df["label"].unique()) if not df.empty else [])

Readable images: 3536
Unreadable images: 0
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']


### What is happening in the next code?

The code counts images by dataset, source, and class, then shows the total number of images in each class.  
It also summarizes the image dimensions, showing statistics such as minimum, maximum, average, and standard deviation.

In [11]:
print("Images by dataset, source, and class:")
display(
    df.groupby(["dataset", "source", "label"])
      .size()
      .rename("images")
      .reset_index()
)

print("Total images per class:")
display(df["label"].value_counts().sort_index())

print("Image dimensions:")
display(df[["width", "height"]].describe())

Images by dataset, source, and class:


,dataset,source,label,images
0,dataset1,test,ambulance,49
1,dataset1,test,autobus,50
2,dataset1,test,kamyun,41
3,dataset1,test,kamyunet,59
4,dataset1,test,minibus,50
5,dataset1,test,savari,50
6,dataset1,test,taxi,50
7,dataset1,test,vanet,50
8,dataset1,train,ambulance,50
9,dataset1,train,autobus,50


Total images per class:


label
ambulance    422
autobus      521
kamyun       437
kamyunet     618
minibus      438
savari       352
taxi         348
vanet        400
Name: count, dtype: int64

Image dimensions:


,width,height
count,3536.000000,3536.000000
mean,258.635747,320.748586
std,72.258108,95.563602
min,117.000000,156.000000
25%,195.000000,252.000000
50%,258.000000,278.000000
75%,313.000000,388.000000
max,666.000000,970.000000


### What is happening in the next code?

The code creates a `data/manifests` folder and saves the image information from `df` into `all_images.csv`.  
It then prints the file location and verifies that the saved CSV contains the expected number of rows.

In [12]:
manifest_dir = PROJECT_ROOT / "data" / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)

manifest_path = manifest_dir / "all_images.csv"
df.to_csv(manifest_path, index=False)

print("Saved:", manifest_path)
print("Rows saved:", len(pd.read_csv(manifest_path)))

Saved: d:\Maktab_Sharif_Projects\traffic-vehicle-classification\data\manifests\all_images.csv
Rows saved: 3536
